In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

# مسار البيانات المعالجة
PROCESSED_DIR = Path("../data/processed")

# 1. قراءة البيانات (Parquet يقرأ المجلد كاملاً كأنه ملف واحد)
print("📂 Loading Processed Data...")
try:
    df = pd.read_parquet(PROCESSED_DIR)
    print(f"✅ Loaded successfully! Shape: {df.shape}")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    # توقف هنا إذا فشل التحميل

# 2. عرض عينة عشوائية
print("\n🔍 Random Sample:")
display(df)

# 3. فحص أنواع البيانات (Data Types)
print("\n📊 Data Types:")
print(df.dtypes)

# 4. فحص عمود القوائم (Diagnoses List) - أهم اختبار!
# نتأكد أنها List حقيقية وليست نصاً
print("\n🧪 Inspecting Diagnoses Lists:")
sample_diag = df['diagnosis_names'].iloc[0]
print(f"Type of cell content: {type(sample_diag)}")
print(f"Content example: {sample_diag}")

if isinstance(sample_diag, np.ndarray) or isinstance(sample_diag, list):
    print("✅ GREAT! Diagnoses are stored as Arrays/Lists.")
else:
    print("⚠️ WARNING: Diagnoses might be stored as Strings!")

# 5. فحص القيم المفقودة (Nulls)
print("\n🕳️ Missing Values Check:")
print(df.isnull().sum())

# 6. فحص المنطق (Business Logic Check)
# هل هناك مريض لديه وصفة دواء ولكن لا يوجد له عمر أو وزن؟ (مشكلة في الـ Join)
orphan_records = df[df['anchor_age'].isnull()]

# 7. فحص التزامن بين الأكواد والأسماء (Alignment Check)
print("\n⚖️ Checking Alignment (Codes vs Names lengths)...")

# دالة لحساب الفرق في الطول
def check_length_mismatch(row):
    return len(row['diagnosis_codes']) != len(row['diagnosis_names'])

# تطبيق الفحص
mismatches = df[df.apply(check_length_mismatch, axis=1)]

if len(mismatches) > 0:
    print(f"⚠️ CRITICAL WARNING: Found {len(mismatches)} rows where Codes count != Names count.")
    display(mismatches[['hadm_id', 'diagnosis_codes', 'diagnosis_names']].head(3))
else:
    print("✅ PERFECT ALIGNMENT: Every diagnosis code has a corresponding name.")
    
if len(orphan_records) > 0:
    print(f"\n⚠️ Alert: Found {len(orphan_records)} prescriptions with missing patient info (Join failed for some IDs).")
else:
    print("\n✅ Integrity Check Passed: All prescriptions link to a patient.")

📂 Loading Processed Data...
✅ Loaded successfully! Shape: (18087, 12)

🔍 Random Sample:


,subject_id,hadm_id,starttime,drug,ndc,dose_val_rx,dose_unit_rx,gender,anchor_age,weight,diagnosis_codes,diagnosis_names
0,10027602,28166872,2201-10-30 12:00:00,Fentanyl Citrate,NaN,None,None,F,71,128.022222,"[5990, 51881, 49390, 79311, 04104, 78722, 5308...","[Urinary tract infection, site not specified, ..."
1,10027602,28166872,2201-10-30 12:00:00,Fentanyl Citrate,NaN,None,None,F,71,128.022222,"[5990, 51881, 49390, 79311, 04104, 78722, 5308...","[Urinary tract infection, site not specified, ..."
2,10027602,28166872,2201-10-31 12:00:00,Lorazepam,NaN,None,None,F,71,128.022222,"[5990, 51881, 49390, 79311, 04104, 78722, 5308...","[Urinary tract infection, site not specified, ..."
3,10027602,28166872,2201-10-30 12:00:00,Midazolam,NaN,None,None,F,71,128.022222,"[5990, 51881, 49390, 79311, 04104, 78722, 5308...","[Urinary tract infection, site not specified, ..."
4,10027602,28166872,2201-10-30 12:00:00,Midazolam,NaN,None,None,F,71,128.022222,"[5990, 51881, 49390, 79311, 04104, 78722, 5308...","[Urinary tract infection, site not specified, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
18082,10038081,20755971,2115-10-11 14:00:00,Artificial Tears,2.305060e+07,1-2,DROP,F,63,139.990000,"[5187, 514, 5723, 40390, V1001, 51881, 7885, 2...",[Transfusion related acute lung injury (TRALI)...
18083,10002428,23473524,2156-05-12 13:00:00,Artificial Tears,2.305060e+07,1-2,DROP,F,80,97.585714,"[V707, 00845, 5990, 53081, 4240, 42843, 2859, ...","[Examination of participant in clinical trial,..."
18084,10040025,27996267,2148-01-26 19:00:00,OxyCODONE (Immediate Release),9.046446e+08,5-10,mg,F,64,210.301000,"[E1122, Z515, I130, B954, E1152, L89899, I2510...",[Type 2 diabetes mellitus with diabetic chroni...
18085,10014354,26228185,2150-05-01 01:00:00,Carbamide Peroxide 6.5%,7.811207e+10,5-10,DROP,M,60,271.868211,"[D696, Z950, D509, E785, F329, R740, T402X5A, ...","[Thrombocytopenia, unspecified, Presence of ca..."



📊 Data Types:
subject_id                  Int64
hadm_id                     Int64
starttime          datetime64[ns]
drug                       object
ndc                       float64
dose_val_rx                object
dose_unit_rx               object
gender                   category
anchor_age                  int64
weight                    float64
diagnosis_codes            object
diagnosis_names            object
dtype: object

🧪 Inspecting Diagnoses Lists:
Type of cell content: <class 'numpy.ndarray'>
Content example: ['Urinary tract infection, site not specified' 'Acute respiratory failure'
 'Asthma, unspecified type, unspecified' 'Solitary pulmonary nodule'
 'Streptococcus infection in conditions classified elsewhere and of unspecified site, streptococcus, group D [Enterococcus]'
 'Dysphagia, oropharyngeal phase' 'Esophageal reflux' 'Hypoxemia'
 'Methicillin resistant pneumonia due to Staphylococcus aureus'
 'Attention deficit disorder with hyperactivity' 'Other convulsions'
 